In [ ]:
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.3 MB/s eta 0:00:00
Libraries ready!


In [ ]:
from groq import Groq
API_KEY="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"
client=Groq(api_key=API_KEY)
MODEL="llama-3.1-8b-instant"
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_Key is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_Key is replaced with your actual key!


In [ ]:
def ask_llm(user_message,system_message="You are a helpful assistant.",
            temperature=0.7,max_tokens=500):
   response=client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
   )
   return response.choices[0].message.content

test_response=ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences"
)
print('=== LLM Response ===')
print(test_response)


=== LLM Response ===
ETL (Extract, Transform, Load) is a data engineering process used to transfer data from various sources, transform it into a standardized format, and load it into a target system such as a data warehouse, database, or data lake. The ETL process involves three main stages: extracting data from sources, transforming the data into a usable format, and loading it into the target system for analysis and reporting.


In [ ]:
response_etl=ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture"
    "(Bronze,silver,Gold layers) related to ETL pipelines.",
    system_message="You are a data engineering instructor."
                   "Be concise and practical"
)
print("Medallion + ETL connection:")
print(response_etl)
print()
print('----Token explanation---')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(-6000 words per conversation)')

Medallion + ETL connection:
Here are 3 key points on how the Medallion Architecture (Bronze, Silver, Gold layers) relates to ETL pipelines:

• **Bronze Layer (Raw Data)**: This layer contains raw, unprocessed data from various sources, such as databases, APIs, or files. The ETL pipeline in this layer is focused on data ingestion, where data is extracted from the source and loaded into a centralized repository, often a data lake, in its raw form.

• **Silver Layer (Processed Data)**: In this layer, data is transformed and enriched to make it more valuable and usable for analysis. The ETL pipeline in this layer performs data cleansing, data normalization, and data aggregation to create a processed dataset that is more suitable for business intelligence and reporting.

• **Gold Layer (Curated Data)**: This layer contains the final, polished data product that is ready for consumption by business users. The ETL pipeline in this layer focuses on data integration, where data from multiple sou

In [ ]:
response_etl=ask_llm(
    "In a Paragraph,explain about the GenAI"
    "concepts that related to the GenAI",
    system_message="You are a GenAI Professional"
                   "Be concise and practical"
)
print("Medallion + ETL connection:")
print(response_etl)
print()
print('----Token explanation---')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(-6000 words per conversation)')

Medallion + ETL connection:
GenAI encompasses several key concepts, including **Generative Models**, which can create new content such as images, text, or music. **Transformers**, a type of neural network architecture that excels at processing sequential data like text or speech. **Self-Supervised Learning**, where AI models learn from raw data without human supervision, enabling them to adapt to new tasks and environments. **Meta-Learning**, which involves training AI models to learn how to learn from few examples, allowing them to quickly adapt to new situations. **Multimodal Learning**, the ability of AI models to process and integrate multiple types of data, such as text, images, and audio. **Explainability**, a crucial aspect of GenAI, which involves understanding how AI models arrive at their decisions and predictions, making them more transparent and reliable. These concepts are driving the development of more sophisticated and flexible AI systems that can learn, reason, and int

In [ ]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigrade Road, Banglore 560025,Karnataka,India"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()

ambiguous_response=ask_llm("Clean this data :ramesh kumar,45000,mumbai")
print('Ambigous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable')

Zero-Shot Result:
The city name is Banglore.

Ambigous Zero-Shot Result:
The data appears to be a record with a name, salary, and location. Here's the cleaned data with proper formatting:

Name: Ramesh Kumar
Salary: ₹45,000 (assuming the currency is Indian Rupees)
Location: Mumbai

In a more structured format, it could look like this:

| Name | Salary | Location |
| --- | --- | --- |
| Ramesh Kumar | ₹45,000 | Mumbai |

Problem: output format is unpredictable and not machine-parseable


In [ ]:
few_shot_prompt = """
Convert the employee text to JSON.Here are examples:
Input: RAMESH KUMAR,45000,mumbai
Output:
{
    "name": "Ramesh kumar",
    "salary": 45000,
    "city": "mumbai"
}

Input: PRIYA SHARMA,55000,chennai
Output:
{
    "name": "Priya Sharma",
    "salary": 55000,
    "city": "Chennai"
}

Now convert this:

Input: ANAYA DAS,60000,delhi
Output:
"""

few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print('Few_Shot Result:')
print(few_shot_response)
print()

try:
  parsed=json.loads(few_shot_response.strip())
  print('Sucessfully parsed JSON!')
  print(f'Name: {parsed["name"]},Salary: {parsed["salary"]} , City: {parsed["city"]}')
except json.JSONDecodeError:
  print('Parsing failed-model added extra text')
  print('Solution:add explicit instructions in the system prompt')

Few_Shot Result:
Here's the Python code to convert the employee text to JSON:

```python
import json

def convert_to_json(name, salary, city):
    # Split the name into first and last names
    names = name.split()
    first_name = ' '.join(names[:-1])
    last_name = names[-1]

    # Capitalize the first letter of each word in the name
    first_name = first_name.title()
    last_name = last_name.title()

    # Create a dictionary with the employee details
    employee = {
        "name": f"{first_name} {last_name}",
        "salary": salary,
        "city": city.title()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
name = "ANAYA DAS"
salary = 60000
city = "delhi"
print(convert_to_json(name, salary, city))
```

Output:
```json
{
    "name": "Anaya Das",
    "salary": 60000,
    "city": "Delhi"
}
```

This code defines a function `convert_to_json` that takes the employee's name, salary, and city

In [ ]:
few_shot_prompt = """
Identify the gender from the person's name and return JSON.

Input: RAMESH KUMAR
Output:
{
    "name": "Ramesh Kumar",
    "gender": "Boy"
}

Input: PRIYA SHARMA
Output:
{
    "name": "Priya Sharma",
    "gender": "Girl"
}

Now convert this:

Input: ANAYA DAS
Output:
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)

print("Few Shot Result:")
print(few_shot_response)
print()

import json

try:
    parsed = json.loads(few_shot_response.strip())
    print("Successfully parsed JSON!")
    print(f"Name: {parsed['name']}")
    print(f"Gender: {parsed['gender']}")
except json.JSONDecodeError:
    print("Parsing failed - model added extra text")
    print("Solution: add explicit instructions in the system prompt")

Few Shot Result:
To identify the gender from the person's name, we can use a combination of machine learning models and natural language processing techniques. However, for simplicity, we can use a basic approach based on common Indian naming conventions.

Here's a Python function that uses this approach:

```python
import json

def identify_gender(name):
    # Split the name into first and last names
    names = name.split()
    
    # Initialize the gender
    gender = "Unknown"
    
    # Check if the name is a common boy's name
    boy_names = ["Ramesh", "Kumar", "Ankit", "Rahul", "Amit", "Vikas", "Ajay", "Suresh", "Rajesh", "Sanjay"]
    if names[0] in boy_names:
        gender = "Boy"
    
    # Check if the name is a common girl's name
    girl_names = ["Priya", "Shruti", "Anushka", "Priyanka", "Shilpa", "Ananya", "Sakshi", "Riya", "Aisha", "Neha"]
    if names[0] in girl_names:
        gender = "Girl"
    
    # If the name is not a common boy's or girl's name, check the suffix

In [ ]:
same_question = "Review this Python code and identify any issues\n"\
"df[total] = df[price] * df[quantity]\n"\
"print(df.groupby('product').sum())\n"

generic_response=ask_llm(same_question,temperature=0.2)
print('without Role Prompting')
print(generic_response[:300],'...')
print()

role_response=ask_llm(same_question,
                      system_message="You are senior data engineer with 10 years of production"
                      "experience. REview code critically for production readiness,"
                      "data types issues,and potential failures at scale.",
                      temperature=0.2)
print('with Role Prompting (Senior Data Engineer):')
print(role_response[:400],'...')
print()
print('Notice:role prompting produces more technical,actionable feedback')

without Role Prompting
The provided Python code appears to be a part of a data analysis task using the pandas library. However, there are a few potential issues:

1. **Undefined variables**: The code uses variables `df`, `total`, `price`, and `quantity`, but it doesn't specify what these variables represent or where they  ...

with Role Prompting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are several potential issues that could impact production readiness:

### 1. Data Type Issues

The code assumes that the `price` and `quantity` columns are numeric, which might not always be the case. If these columns contain non-numeric data, the multiplication operatio ...

Notice:role prompting produces more technical,actionable feedback


In [ ]:
prompt="Give me one creative name for a data analytics startup"

print("===Temperature Experiment===")
for temp in[0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature={temp}: {response.strip()}')
  time.sleep(1)

print()
print('Observation:')
print(' temperature=0.0->same or very similar answer every run(deterministic)')
print(' temperature=0.5->some variation')
print(' temperature=1.0->more creative/varied,sometimes surpising')
print()
print('Rule for data engineering tasks:use temperature=0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE output _not creative variation')

===Temperature Experiment===
Temperature=0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys a sense of innovation and forward-thinking, which is perfect for a data analytics startup.
Temperature=0.5: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying that the startup helps clients connect the dots between data points to gain valuable insights. This name conveys a sense of innovative problem-solving and forward-thinking, which is perfect for a data analytics startup.
Temperature=1.0: Here's a creative name for a data analytics startup:

**Nexa Vista**

"Nexa" implies connections and intersections of data, while "Vista" suggests a broad, panoramic view of the insights and trends revealed through analytics. This name has a modern and sleek sound to i

In [ ]:
messy_invoices=[
    "INV-2014-0891 TECHWORLD SOLUTIONS 15th jan 2024 Rs. 45,000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for Office cleaning services",
    "INV-2014-103 | arjun nair consultancy |8000 | march 15 2024| python training",
    "SURESH RAO HARDWARE STORE 25000 Keyword and Mouse accessories 2024/01/20",
    "Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000| Server hardware"
]

print('Messy Invoices to process:')
for i, inv in enumerate(messy_invoices,1):
  print(f'{i+1}.{inv}')
print(f'\nTotal: {len(messy_invoices)} invoices')


Messy Invoices to process:
2.INV-2014-0891 TECHWORLD SOLUTIONS 15th jan 2024 Rs. 45,000 Laptop purchase
3.Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for Office cleaning services
4.INV-2014-103 | arjun nair consultancy |8000 | march 15 2024| python training
5.SURESH RAO HARDWARE STORE 25000 Keyword and Mouse accessories 2024/01/20
6.Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000| Server hardware

Total: 5 invoices


In [ ]:
import json

few_shot_prompt = '''
Extract student name and marks from the text and represent it in JSON format.

Here are examples:

Input: "Mathi scored 95 marks"
Output: {"name": "Mathi", "marks": 95}

Input: "Monika scored 88 marks"
Output: {"name": "Monika", "marks": 88}

Now extract from:

Input: "Shannu scored 91 marks"
Output:
'''

few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)

print("==== Few Shot Result ====")
print(few_shot_response)
print()

try:
    parsed = json.loads(few_shot_response.strip())
    print("Successfully parsed JSON!")
    print(f"Name: {parsed['name']}")
    print(f"Marks: {parsed['marks']}")
except json.JSONDecodeError:
    print("Parsing failed - model added extra text")
    print("Solution: Add explicit instructions in the system prompt")

==== Few Shot Result ====
Input: "Shannu scored 91 marks"

To extract the student name and marks, I will use the following Python code:

```python
import re

def extract_student_info(text):
    pattern = r"(\w+) scored (\d+) marks"
    match = re.search(pattern, text)
    if match:
        name = match.group(1)
        marks = int(match.group(2))
        return {"name": name, "marks": marks}
    else:
        return None

text = "Shannu scored 91 marks"
print(extract_student_info(text))
```

Output:
```python
{'name': 'Shannu', 'marks': 91}
```

This code uses a regular expression to match the pattern of the input string. The `(\w+)` part matches one or more word characters (letters, numbers, or underscores) to capture the student's name, and the `(\d+)` part matches one or more digits to capture the marks. The `re.search` function returns a match object if the pattern is found in the string, and the `group` method is used to extract the matched groups. The result is then returned as a

In [ ]:
print("Processing Invoices with LLM...")
print("="*100)

user_invoice_message = "\n".join(messy_invoices)

extracted_records = ask_llm(user_invoice_message,
                            system_message='''Clean the data and organize data in same structure
                            (company,amount, invoice_date, product) return as a DataFrame.
                            print the output DataFrame.
                            No code.''')
print("==== Sample Response ====")
print(extracted_records,'\n')

Processing Invoices with LLM...
==== Sample Response ====
Here's a step-by-step process to clean and organize the data:

1. **Extract Company Names**: 
   - TECHWORLD SOLUTIONS
   - PRIYA ENTERPRISES
   - arjun nair consultancy
   - SURESH RAO HARDWARE STORE
   - Ananya Tech Solutions

2. **Extract Invoice Dates**:
   - 15th jan 2024
   - 07-02-2024
   - march 15 2024
   - 2024/01/20
   - 28-feb-24

3. **Extract Invoice Amounts**:
   - Rs. 45,000
   - 12500
   - 8000
   - 25000
   - INR 95,000

4. **Extract Product/Service Names**:
   - Laptop purchase
   - Office cleaning services
   - python training
   - Keyword and Mouse accessories
   - Server hardware

5. **Organize Data into a DataFrame**:
   - Create a list of company names, invoice amounts, invoice dates, and product/service names.
   - Use a list comprehension to create the DataFrame. Each list will represent a row in the DataFrame.

Here's the organized data:

| Company                         | Amount | Invoice Date       |

In [ ]:
extracted_records = [
    {
        "invoice_number": "INV001",
        "amount": "2500",
        "invoice_date": "2025-06-01"
    },
    {
        "invoice_number": "INV002",
        "amount": "1800",
        "invoice_date": "2025-06-02"
    },
    {
        "invoice_number": "INV003",
        "amount": "3200",
        "invoice_date": "2025-06-03"
    }
]

invoices_df = pd.DataFrame(extracted_records)

invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)

invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['invoice_date'],
    errors='coerce'
)

print("=== SMART DATA CLEANER OUTPUT ===")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}\n")

print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 3 | Columns: 3

invoice_number  amount invoice_date
        INV001    2500   2025-06-01
        INV002    1800   2025-06-02
        INV003    3200   2025-06-03


In [ ]:
messy_invoices = [
    "INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase",
    "Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training",
    "SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10",
    "Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware"
]

print('MEssy invoices to process:')
for i, inv in enumerate(messy_invoices,1):
  print(f'{i+1}.{inv}')
print(f'\nTotal: {len(messy_invoices)}invoices')

MEssy invoices to process:
2.INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase
3.Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services
4.#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training
5.SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10
6.Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware

Total: 5invoices


In [ ]:
import json

print("Processing Invoices with LLM...")
print("="*100)

user_invoice_message = "\n".join(messy_invoices)

llm_raw_response = ask_llm(user_invoice_message,
                            system_message='''Clean the data and organize data in same structure
                            (company,amount, invoice_date, product)
                            return as JSON.
                            Only output the JSON.
                            No code.''')

print("==== Raw LLM Response ====")
print(llm_raw_response,'\n')

try:
  extracted_records = json.loads(llm_raw_response)
  print("Successfully parsed LLM response to a list of dictionaries.")
except json.JSONDecodeError as e:
  print(f"Error decoding JSON from LLM response: {e}")
  print("LLM Raw Response (unparseable):\n", llm_raw_response)
  # Set extracted_records to an empty list or handle as appropriate if parsing fails
  extracted_records = []
except Exception as e:
  print(f"An unexpected error occurred during parsing: {e}")
  extracted_records = []

Processing Invoices with LLM...
==== Raw LLM Response ====
[
  {
    "company": "Techworld Solutions",
    "amount": 45000,
    "invoice_date": "2024-01-15",
    "product": "Laptop"
  },
  {
    "company": "Priya Entireprices",
    "amount": 12500,
    "invoice_date": "2024-02-07",
    "product": "Office Cleaning services"
  },
  {
    "company": "Arjun Nair Consultancy",
    "amount": 5000,
    "invoice_date": "2024-03-15",
    "product": "Python training"
  },
  {
    "company": "Suresh Rao Hardware Store",
    "amount": 2500,
    "invoice_date": "2024-01-10",
    "product": "Keyboard and mouse accessories"
  },
  {
    "company": "Ananya Tech Solutions",
    "amount": 95000,
    "invoice_date": "2024-02-28",
    "product": "Server hardware"
  }
] 

Successfully parsed LLM response to a list of dictionaries.
